# Fatoração de Cholesky (Algoritmo Adaptado do Slide)

Objetivo: dada uma matriz simétrica definida positiva $A\in\mathbb{R}^{n\times n}$, encontrar $G$ triangular inferior tal que:
$$A = G\,G^T$$
O algoritmo abaixo segue exatamente a estrutura apresentada no slide (índices do slide começam em 1; aqui convertemos para índices Python iniciando em 0).

Etapas principais:
1. Calcular $g_{11} = \sqrt{a_{11}}$.
2. Para $i=2..n$: $g_{i1} = a_{i1} / g_{11}$.
3. Para cada coluna $k = 2..n-1$:
   - Somatório 1: $\text{soma} = \sum_{j=1}^{k-1} g_{kj}^2$
   - $r = a_{kk} - \text{soma}$ e $g_{kk} = \sqrt{r}$
   - Para $i = k+1..n$:
       * Somatório 2: $\text{soma} = \sum_{j=1}^{k-1} g_{ij} g_{kj}$
       * $g_{ik} = (a_{ik} - \text{soma}) / g_{kk}$
4. Última linha:
   - $\text{soma} = \sum_{j=1}^{n-1} g_{nj}^2$
   - $g_{nn} = \sqrt{a_{nn} - \text{soma}}$

Observação: se algum $r \le 0$, a matriz não é definida positiva (ou há erro numérico).

In [ ]:
import numpy as np
import math

def cholesky_slide(A):
    """
    Fatoração de Cholesky
    ----------
    A : ndarray (n x n)
        Matriz simétrica definida positiva.
    Retorno
    -------
    G : ndarray (n x n)
        Matriz triangular inferior tal que A = G @ G.T
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    G = np.zeros_like(A)
    # Passo 1: g11
    G[0,0] = math.sqrt(A[0,0])
    # Passo 2: primeira coluna (i = 2..n no slide -> i=1..n-1 em Python)
    for i in range(1, n):
        G[i,0] = A[i,0] / G[0,0]
    # Passo 3: colunas k = 2..n-1  (slide) => k = 1..n-2 (Python)
    for k in range(1, n-1):
        # soma = sum_{j=1}^{k-1} g_{k,j}^2  (slide, j=1..k-1) -> j=0..k-1 em Python
        soma = 0.0
        for j in range(0, k):
            soma += G[k,j]**2
        r = A[k,k] - soma
        if r <= 0:
            raise ValueError(f"Matriz não é definida positiva (r={r} <= 0) na posição k={k}.")
        G[k,k] = math.sqrt(r)
        # Linhas i = k+1..n  -> i = k+1..n-1
        for i in range(k+1, n):
            soma = 0.0
            for j in range(0, k):
                soma += G[i,j] * G[k,j]
            G[i,k] = (A[i,k] - soma)/G[k,k]
    # Passo 4: última linha g_nn
    soma = 0.0
    for j in range(0, n-1):
        soma += G[n-1, j]**2
    r = A[n-1, n-1] - soma
    if r <= 0:
        raise ValueError(f"Matriz não é definida positiva (r={r} <= 0) na última etapa.")
    G[n-1, n-1] = math.sqrt(r)
    return G

# Exemplo de teste com matriz SPD (simétrica definida positiva)
# Construímos A = B B^T para garantir SPD
B = np.array([[1,0,0],
              [2,1,0],
              [2,-3,1]], dtype=float)
A = B @ B.T
G = cholesky_slide(A)
print("A =\n", A)
print("G =\n", G)
print("Verificação (G G^T) =\n", G @ G.T)
print("Erro máximo:", np.max(np.abs(A - G @ G.T)))

A =
 [[ 1.  2.  2.]
 [ 2.  5.  1.]
 [ 2.  1. 14.]]
G =
 [[ 1.  0.  0.]
 [ 2.  1.  0.]
 [ 2. -3.  1.]]
Verificação (G G^T) =
 [[ 1.  2.  2.]
 [ 2.  5.  1.]
 [ 2.  1. 14.]]
Erro máximo: 0.0


In [2]:
import numpy as np
def resol_sist_triang_inf(a, b):
    """
    Resolve um sistema de equações lineares Ax = b,
    onde A é uma matriz triangular inferior.
    Parâmetros:
    a : array_like
        Matriz dos coeficientes (triangular inferior).
    b : array_like
        Vetor dos termos independentes.
    """
    n = len(b)
    x = np.zeros(n)
    x[0] = b[0] / a[0, 0]
    for k in range(1, n):
        s = 0
        for j in range(k):
            s += a[k, j] * x[j]
        x[k] = (b[k] - s) / a[k, k]
    return x

# Exemplo de uso
A = np.array([[1, 0, 0],
              [2, 1, 0],
              [2, -3, 1]], dtype=float)

b = np.array([-3, -1, -23], dtype=float)
x = resol_sist_triang_inf(A, b)
print("Sol:", x)

Sol: [-3.  5. -2.]
